In [2]:
from pymongo import MongoClient

# 1. Conectar al contenedor de Mongo
client = MongoClient("database", 27017)

# 2. Crear (o usar) la base de datos y la colección
db = client['ArriendosPrueba_IVRegión'] 
coleccion = db['G2_Intentos'] # Ideal el nombre del grupo ejem: 'G_Ecommerce_scraping'

print("Conexión exitosa a MongoDB")

Conexión exitosa a MongoDB


In [3]:
# ==============================================================================
# PROYECTO BIG DATA - GRUPO 2 (REAL ESTATE)
# Módulo de Extracción Profunda: GoPlaceIt (Arriendos en Coquimbo / La Serena)
# ==============================================================================

import os
import time
import re
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException

# --- 🎯 ASIGNACIÓN DE PÁGINAS PARA EL GRUPO ---
# Estudiante 1: Pone INICIO = 1, FIN = 5
# Estudiante 2: Pone INICIO = 6, FIN = 10
PAGINA_INICIO = 1  
PAGINA_FIN = 1  
# ----------------------------------------------

os.environ["DISPLAY"] = ":99"  
os.system("pkill -9 chrome && pkill -9 chromedriver")
os.system("rm -rf /tmp/.com.google.Chrome.* && rm -rf /tmp/.org.chromium.Chromium.*")
print(f"🧹 Entorno virtual listo. Asignación GoPlaceIt: Páginas {PAGINA_INICIO} a la {PAGINA_FIN}.")

options = Options()
# options.add_argument("--headless=new")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0")

catalogo_urls = []
datos_finales = []

try:
    driver = webdriver.Chrome(options=options)
   
    # ==============================================================================
    # FASE 1: RECOLECCIÓN EN EL CATÁLOGO (ARRIENDOS)
    # ==============================================================================
    # URL oficial de arriendos proporcionada
    url_base = "https://www.goplaceit.com/cl/departamento/arriendo/region/4-region-de-coquimbo"

    for pagina_actual in range(PAGINA_INICIO, PAGINA_FIN + 1):
        print(f"\n--- 📄 [FASE 1] Recolectando enlaces de la Página {pagina_actual} ---")
       
        # Paginación de GoPlaceIt
        url_pagina = f"{url_base}&page={pagina_actual}" if "?" in url_base else f"{url_base}?page={pagina_actual}"
        driver.get(url_pagina)

        try:
            # El Vigía: Esperamos por la clase estable 'featured card'
            WebDriverWait(driver, 15).until(
                EC.presence_of_all_elements_located((By.CSS_SELECTOR, "a[href*='/propiedad/']"))
            )
           
           
            # Scroll Humano
            for _ in range(6):
                driver.execute_script("window.scrollBy(0, 800);")
                time.sleep(1)
            time.sleep(2)

            # Atrapamos los contenedores
            bloques = driver.find_elements(By.CSS_SELECTOR, "a[href*='/propiedad/']")
           
            for bloque in bloques:
                try:
                    url = bloque.get_attribute("href")
                    if not url or "javascript" in url: continue

                    # Leemos el texto de la tarjeta para filtrar por ubicación
                    texto_tarjeta = bloque.get_attribute("textContent").lower()
                   
                    if "coquimbo" in texto_tarjeta:
                        ubicacion = "Coquimbo"      
                    elif "la serena" in texto_tarjeta:
                        ubicacion = "La Serena"    
                    else:
                        continue

                    catalogo_urls.append({
                        "url": url,
                        "ubicacion": ubicacion
                    })
                except:
                    continue

            print(f"✅ Se recolectaron enlaces en la página {pagina_actual}.")

        except TimeoutException:
            print("🚨 Advertencia: GoPlaceIt tardó en cargar. Posible fin de resultados.")
            continue
           
    print(f"\n✅ Catálogo listo: {len(catalogo_urls)} propiedades encontradas. Iniciando Fase 2...")

    # ==============================================================================
    # FASE 2: INSPECCIÓN PROFUNDA POR PROPIEDAD
    # ==============================================================================
    for i, propiedad in enumerate(catalogo_urls):
        # Latido del corazón: Imprime progreso cada 5 departamentos
        if (i + 1) % 5 == 0:
            print(f"⏳ Progreso: Inspeccionando propiedad {i + 1} de {len(catalogo_urls)}...")
       
        registro = {
            "responsable": "Constanza Torres", # Cambiar por el nombre del estudiante
            "fecha_extraccion": time.strftime("%Y-%m-%d %H:%M:%S"),
            "titulo": "Propiedad GoPlaceIt",
            "ubicacion": propiedad["ubicacion"],
            "m2": 0,
            "precio": 0.0,
            "dormitorios": 0,
            "baños": 0,
            "estacionamiento": 0,
            "piscina": 0,
            "quincho": 0,
            "terraza": 0,
            "gimnasio": 0,
            "lavanderia": 0,
            "url": propiedad["url"]
        }

        try:
            driver.get(propiedad["url"])
           
            WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.TAG_NAME, "body")))
            time.sleep(2)
           
            # --- RASPADO MASIVO ---
            texto_total = driver.find_element(By.TAG_NAME, "body").get_attribute("textContent").lower()
           
            # Título
            try:
                registro["titulo"] = driver.find_element(By.TAG_NAME, "h1").get_attribute("textContent").strip()
            except: pass

            # --- EXTRACCIÓN DE PRECIO (Optimizado para Arriendos) ---
            try:
                # Buscamos el bloque de precio principal
                precio_bloque = driver.find_element(By.CSS_SELECTOR, ".gpi-featured-card-price-lg").get_attribute("textContent")
                # Limpiamos el texto, asumiendo formato en pesos (ej: $ 450.000)
                v_limpio = precio_bloque.lower().replace("$", "").replace("clp", "").replace(".", "").replace(",", "").replace("uf", "").strip()
                # Extraemos solo los primeros números seguidos que encontremos
                numeros_precio = re.findall(r'\d+', v_limpio)
                if numeros_precio:
                    registro["precio"] = float(numeros_precio[0])
            except:
                pass

            # --- EXTRACCIÓN NUMÉRICA (RegEx) ---
            match_m2 = re.search(r'(\d+)\s*(m2|m²|mts)', texto_total)
            if match_m2: registro["m2"] = int(match_m2.group(1))

            match_dorm = re.search(r'(\d+)\s*(habitacion|habitaciones|dormitorio)', texto_total)
            if match_dorm: registro["dormitorios"] = int(match_dorm.group(1))

            match_bano = re.search(r'(\d+)\s*baño', texto_total)
            if match_bano: registro["baños"] = int(match_bano.group(1))
           
            match_estac = re.search(r'(\d+)\s*estacionamiento', texto_total)
            if match_estac:
                registro["estacionamiento"] = int(match_estac.group(1))
            elif "estacionamiento" in texto_total:
                registro["estacionamiento"] = 1

            # --- AMENIDADES BOOLEANAS ---
            if "piscina" in texto_total: registro["piscina"] = 1
            if "quincho" in texto_total: registro["quincho"] = 1
            if "terraza" in texto_total: registro["terraza"] = 1
            if "gimnasio" in texto_total: registro["gimnasio"] = 1
            if "lavander" in texto_total: registro["lavanderia"] = 1

            datos_finales.append(registro)

        except Exception as e:
            # Si falla la lectura profunda, guardamos con valores base para no perder la URL
            datos_finales.append(registro)
           
        time.sleep(2)

except Exception as e:
    print(f"🚨 Error crítico en Selenium: {e}")

finally:
    if driver is not None: driver.quit()

# ==============================================================================
# GUARDADO EN MONGODB (UPSERT)
# ==============================================================================
print("\n--- 💾 Iniciando guardado en Base de Datos ---")
try:
    if datos_finales:
        # Asegúrate de usar localhost para evitar el error de conexión
        client = MongoClient("database", 27017, serverSelectionTimeoutMS=5000)
        db = client["ArriendosPrueba_IVRegión"]
        coleccion = db["G2_Intentos"]

        registros_nuevos, registros_actualizados = 0, 0

        for d in datos_finales:
            resultado = coleccion.update_one({"url": d["url"]}, {"$set": d}, upsert=True)
            if resultado.upserted_id: registros_nuevos += 1
            else: registros_actualizados += 1

        print(f"🎉 ¡Extracción GoPlaceIt Arriendos Completada, {datos_finales[0]['responsable']}!")
        print(f"   -> {registros_nuevos} propiedades nuevas.")
        print(f"   -> {registros_actualizados} propiedades actualizadas.")
    else:
        print("⚠️ No se extrajeron datos válidos.")
except Exception as e:
    print(f"🚨 Error de Base de Datos: {e}")

🧹 Entorno virtual listo. Asignación GoPlaceIt: Páginas 1 a la 1.

--- 📄 [FASE 1] Recolectando enlaces de la Página 1 ---
✅ Se recolectaron enlaces en la página 1.

✅ Catálogo listo: 6 propiedades encontradas. Iniciando Fase 2...
⏳ Progreso: Inspeccionando propiedad 5 de 6...

--- 💾 Iniciando guardado en Base de Datos ---
🎉 ¡Extracción GoPlaceIt Arriendos Completada, Constanza Torres!
   -> 6 propiedades nuevas.
   -> 0 propiedades actualizadas.


In [4]:
# ==============================================================================
# SPARK SESSION - ANALISIS DATOS INMOBILIARIOS
# ==============================================================================

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, count, when

# =========================
# SPARK SESSION
# =========================
spark = SparkSession.builder \
    .appName("Analisis_Inmobiliario_GoPlaceIt") \
    .config("spark.mongodb.read.connection.uri", "mongodb://database:27017") \
    .config("spark.mongodb.write.connection.uri", "mongodb://database:27017") \
    .config("spark.mongodb.read.database", "ArriendosPrueba_IVRegión") \
    .config("spark.mongodb.read.collection", "G2_Intentos") \
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:3.0.1") \
    .getOrCreate()

print("Spark iniciado correctamente")

Spark iniciado correctamente


In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Analisis_Inmobiliario_GoPlaceIt") \
    .config("spark.mongodb.read.connection.uri", "mongodb://database:27017/ArriendosPrueba_IVRegión.G2_Intentos") \
    .config("spark.mongodb.write.connection.uri", "mongodb://database:27017/ArriendosPrueba_IVRegión.G2_Intentos") \
    .config("spark.jars.packages",
            "org.mongodb.spark:mongo-spark-connector_2.12:10.3.0") \
    .getOrCreate()

spark

In [3]:
df = spark.read.format("mongodb").load()

df.show(5)

+--------------------+-----+-----------+---------------+-------------------+--------+----------+---+-------+--------+-------+----------------+-------+-------------------+---------+--------------------+
|                 _id|baños|dormitorios|estacionamiento|   fecha_extraccion|gimnasio|lavanderia| m2|piscina|  precio|quincho|     responsable|terraza|             titulo|ubicacion|                 url|
+--------------------+-----+-----------+---------------+-------------------+--------+----------+---+-------+--------+-------+----------------+-------+-------------------+---------+--------------------+
|69e6ed26515bfd345...|    1|          3|              1|2026-04-21 03:20:38|       0|         1| 57|      1|460000.0|      1|Constanza Torres|      0|Propiedad GoPlaceIt|La Serena|https://www.gopla...|
|69e6ed26515bfd345...|    1|          3|              1|2026-04-21 03:20:44|       0|         1| 56|      0| 45000.0|      0|Constanza Torres|      0|Propiedad GoPlaceIt|La Serena|https://www.

In [5]:
df = df.select(
    "responsable",
    "fecha_extraccion",
    "titulo",
    "ubicacion",
    "precio",
    "m2",
    "dormitorios",
    "baños",
    "estacionamiento",
    "piscina",
    "quincho",
    "terraza",
    "gimnasio",
    "lavanderia",
    "url"
)

df.printSchema()
df.show(10)


root
 |-- responsable: string (nullable = true)
 |-- fecha_extraccion: string (nullable = true)
 |-- titulo: string (nullable = true)
 |-- ubicacion: string (nullable = true)
 |-- precio: double (nullable = true)
 |-- m2: integer (nullable = true)
 |-- dormitorios: integer (nullable = true)
 |-- baños: integer (nullable = true)
 |-- estacionamiento: integer (nullable = true)
 |-- piscina: integer (nullable = true)
 |-- quincho: integer (nullable = true)
 |-- terraza: integer (nullable = true)
 |-- gimnasio: integer (nullable = true)
 |-- lavanderia: integer (nullable = true)
 |-- url: string (nullable = true)

+----------------+-------------------+-------------------+---------+--------+---+-----------+-----+---------------+-------+-------+-------+--------+----------+--------------------+
|     responsable|   fecha_extraccion|             titulo|ubicacion|  precio| m2|dormitorios|baños|estacionamiento|piscina|quincho|terraza|gimnasio|lavanderia|                 url|
+----------------+--

In [7]:
from pyspark.sql.functions import col, avg, count, when, desc

df = df.withColumn("precio", col("precio").cast("double")) \
       .withColumn("m2", col("m2").cast("int")) \
       .withColumn("dormitorios", col("dormitorios").cast("int")) \
       .withColumn("baños", col("baños").cast("int"))


In [8]:
from pyspark.sql.functions import count, avg, round

print("--- IMPACTO DE AMENIDADES POR CIUDAD ---")

amenidades = [
    "piscina",
    "quincho",
    "gimnasio",
    "estacionamiento",
    "terraza",
    "lavanderia"
]

for am in amenidades:
    
    print(f"\n Análisis de: {am.upper()}")

    df.groupBy("ubicacion", am).agg(
        count("*").alias("Cantidad"),
        round(avg("precio"), 0).alias("Precio_Promedio")
    ).orderBy("ubicacion", ascending=True).show()

--- IMPACTO DE AMENIDADES POR CIUDAD ---

 Análisis de: PISCINA
+---------+-------+--------+---------------+
|ubicacion|piscina|Cantidad|Precio_Promedio|
+---------+-------+--------+---------------+
| Coquimbo|      0|       1|            0.0|
|La Serena|      1|       2|       440000.0|
|La Serena|      0|       3|       213333.0|
+---------+-------+--------+---------------+


 Análisis de: QUINCHO
+---------+-------+--------+---------------+
|ubicacion|quincho|Cantidad|Precio_Promedio|
+---------+-------+--------+---------------+
| Coquimbo|      0|       1|            0.0|
|La Serena|      1|       2|       440000.0|
|La Serena|      0|       3|       213333.0|
+---------+-------+--------+---------------+


 Análisis de: GIMNASIO
+---------+--------+--------+---------------+
|ubicacion|gimnasio|Cantidad|Precio_Promedio|
+---------+--------+--------+---------------+
| Coquimbo|       0|       1|            0.0|
|La Serena|       1|       1|       420000.0|
|La Serena|       0|       

In [1]:
from pymongo import MongoClient

# 1. Conectar al contenedor de Mongo
client = MongoClient("database", 27017)

# 2. Crear (o usar) la base de datos y la colección
db = client['ArriendosPrueba_IVRegión'] 
coleccion = db['G2_IntentoGoplaceit'] # Ideal el nombre del grupo ejem: 'G_Ecommerce_scraping'

print("Conexión exitosa a MongoDB")

Conexión exitosa a MongoDB


In [1]:
# ==============================================================================
# PROYECTO BIG DATA - GRUPO 2 (REAL ESTATE)
# Módulo de Extracción Profunda: GoPlaceIt (Arriendos en Coquimbo / La Serena)
# ==============================================================================

import os
import time
import re
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException

# --- 🎯 ASIGNACIÓN DE PÁGINAS PARA EL GRUPO ---
# Estudiante 1: Pone INICIO = 1, FIN = 5
# Estudiante 2: Pone INICIO = 6, FIN = 10
PAGINA_INICIO = 1  
PAGINA_FIN = 1  
# ----------------------------------------------

os.environ["DISPLAY"] = ":99"  
os.system("pkill -9 chrome && pkill -9 chromedriver")
os.system("rm -rf /tmp/.com.google.Chrome.* && rm -rf /tmp/.org.chromium.Chromium.*")
print(f"🧹 Entorno virtual listo. Asignación GoPlaceIt: Páginas {PAGINA_INICIO} a la {PAGINA_FIN}.")

options = Options()
# options.add_argument("--headless=new")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0")

catalogo_urls = []
datos_finales = []

try:
    driver = webdriver.Chrome(options=options)
   
    # ==============================================================================
    # FASE 1: RECOLECCIÓN EN EL CATÁLOGO (ARRIENDOS)
    # ==============================================================================
    # URL oficial de arriendos proporcionada
    url_base = "https://www.goplaceit.com/cl/departamento/arriendo/region/4-region-de-coquimbo"

    for pagina_actual in range(PAGINA_INICIO, PAGINA_FIN + 1):
        print(f"\n--- 📄 [FASE 1] Recolectando enlaces de la Página {pagina_actual} ---")
       
        # Paginación de GoPlaceIt
        url_pagina = f"{url_base}&page={pagina_actual}" if "?" in url_base else f"{url_base}?page={pagina_actual}"
        driver.get(url_pagina)

        try:
            # El Vigía: Esperamos por la clase estable 'featured card'
            WebDriverWait(driver, 15).until(
                EC.presence_of_all_elements_located((By.CSS_SELECTOR, "a[href*='/propiedad/']"))
            )
           
           
            # Scroll Humano
            for _ in range(6):
                driver.execute_script("window.scrollBy(0, 800);")
                time.sleep(1)
            time.sleep(2)

            # Atrapamos los contenedores
            bloques = driver.find_elements(By.CSS_SELECTOR, "a[href*='/propiedad/']")
           
            for bloque in bloques:
                try:
                    url = bloque.get_attribute("href")
                    if not url or "javascript" in url: continue

                    # Leemos el texto de la tarjeta para filtrar por ubicación
                    texto_tarjeta = bloque.get_attribute("textContent").lower()
                   
                    if "coquimbo" in texto_tarjeta:
                        ubicacion = "Coquimbo"      
                    elif "la serena" in texto_tarjeta:
                        ubicacion = "La Serena"    
                    else:
                        continue

                    catalogo_urls.append({
                        "url": url,
                        "ubicacion": ubicacion
                    })
                except:
                    continue

            print(f"✅ Se recolectaron enlaces en la página {pagina_actual}.")

        except TimeoutException:
            print("🚨 Advertencia: GoPlaceIt tardó en cargar. Posible fin de resultados.")
            continue
           
    print(f"\n✅ Catálogo listo: {len(catalogo_urls)} propiedades encontradas. Iniciando Fase 2...")

    # ==============================================================================
    # FASE 2: INSPECCIÓN PROFUNDA POR PROPIEDAD
    # ==============================================================================
    for i, propiedad in enumerate(catalogo_urls):
        # Latido del corazón: Imprime progreso cada 5 departamentos
        if (i + 1) % 5 == 0:
            print(f"⏳ Progreso: Inspeccionando propiedad {i + 1} de {len(catalogo_urls)}...")
       
        registro = {
            "responsable": "Constanza Torres", # Cambiar por el nombre del estudiante
            "fecha_extraccion": time.strftime("%Y-%m-%d %H:%M:%S"),
            "titulo": "Propiedad GoPlaceIt",
            "ubicacion": propiedad["ubicacion"],
            "m2": 0,
            "precio": 0.0,
            "dormitorios": 0,
            "baños": 0,
            "estacionamiento": 0,
            "piscina": 0,
            "quincho": 0,
            "terraza": 0,
            "gimnasio": 0,
            "lavanderia": 0,
            "url": propiedad["url"]
        }

        try:
            driver.get(propiedad["url"])
           
            WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.TAG_NAME, "body")))
            time.sleep(2)
           
            # --- RASPADO MASIVO ---
            texto_total = driver.find_element(By.TAG_NAME, "body").get_attribute("textContent").lower()


            # --- FILTRO ARRIENDO DIARIO ---
            #palabras_descartar = [
             #   "por día",
              #  "por dia",
               # "diario",
                #"por noche",
                #"noche",
                #"vacaciones",
                #"temporada",
                #"amoblado diario",
                #"diaria"
            #]

            #titulo = registro.get("titulo", "").lower()

            #if any(p in titulo for p in palabras_descartar):
            #continue
           
            # Título
            try:
                registro["titulo"] = driver.find_element(By.TAG_NAME, "h1").get_attribute("textContent").strip()
            except: pass

            # --- EXTRACCIÓN DE PRECIO (Optimizado para Arriendos) ---
            try:
                # Buscamos el bloque de precio principal
                precio_bloque = driver.find_element(By.CSS_SELECTOR, ".gpi-featured-card-price-lg").get_attribute("textContent")
                # Limpiamos el texto, asumiendo formato en pesos (ej: $ 450.000)
                v_limpio = precio_bloque.lower().replace("$", "").replace("clp", "").replace(".", "").replace(",", "").replace("uf", "").strip()
                # Extraemos solo los primeros números seguidos que encontremos
                numeros_precio = re.findall(r'\d+', v_limpio)
                if numeros_precio:
                    registro["precio"] = float(numeros_precio[0])
            except:
                pass
                # --- FILTRO PRECIO SOSPECHOSO ---
            if registro["precio"] != 0 and registro["precio"] < 150000:
                print("Descartado precio sospechoso:", registro["precio"], registro["url"])
                continue

            # --- EXTRACCIÓN NUMÉRICA (RegEx) ---
            match_m2 = re.search(r'(\d+)\s*(m2|m²|mts)', texto_total)
            if match_m2: registro["m2"] = int(match_m2.group(1))

            match_dorm = re.search(r'(\d+)\s*(habitacion|habitaciones|dormitorio)', texto_total)
            if match_dorm: registro["dormitorios"] = int(match_dorm.group(1))

            match_bano = re.search(r'(\d+)\s*baño', texto_total)
            if match_bano: registro["baños"] = int(match_bano.group(1))
           
            match_estac = re.search(r'(\d+)\s*estacionamiento', texto_total)
            if match_estac:
                registro["estacionamiento"] = int(match_estac.group(1))
            elif "estacionamiento" in texto_total:
                registro["estacionamiento"] = 1

            # --- AMENIDADES BOOLEANAS ---
            if "piscina" in texto_total: registro["piscina"] = 1
            if "quincho" in texto_total: registro["quincho"] = 1
            if "terraza" in texto_total: registro["terraza"] = 1
            if "gimnasio" in texto_total: registro["gimnasio"] = 1
            if "lavander" in texto_total: registro["lavanderia"] = 1

            datos_finales.append(registro)

        except Exception as e:
            # Si falla la lectura profunda, guardamos con valores base para no perder la URL
            datos_finales.append(registro)
           
        time.sleep(2)

except Exception as e:
    print(f"🚨 Error crítico en Selenium: {e}")

finally:
    if driver is not None: driver.quit()

# ==============================================================================
# GUARDADO EN MONGODB (UPSERT)
# ==============================================================================
print("\n--- 💾 Iniciando guardado en Base de Datos ---")
try:
    if datos_finales:
        # Asegúrate de usar localhost para evitar el error de conexión
        client = MongoClient("database", 27017, serverSelectionTimeoutMS=5000)
        db = client["ArriendosPrueba_IVRegión"]
        coleccion = db["G2_IntentoGoplaceit"]

        registros_nuevos, registros_actualizados = 0, 0

        for d in datos_finales:
            resultado = coleccion.update_one({"url": d["url"]}, {"$set": d}, upsert=True)
            if resultado.upserted_id: registros_nuevos += 1
            else: registros_actualizados += 1

        print(f"🎉 ¡Extracción GoPlaceIt Arriendos Completada, {datos_finales[0]['responsable']}!")
        print(f"   -> {registros_nuevos} propiedades nuevas.")
        print(f"   -> {registros_actualizados} propiedades actualizadas.")
    else:
        print("⚠️ No se extrajeron datos válidos.")
except Exception as e:
    print(f"🚨 Error de Base de Datos: {e}")

🧹 Entorno virtual listo. Asignación GoPlaceIt: Páginas 1 a la 1.

--- 📄 [FASE 1] Recolectando enlaces de la Página 1 ---
✅ Se recolectaron enlaces en la página 1.

✅ Catálogo listo: 30 propiedades encontradas. Iniciando Fase 2...
Descartado precio sospechoso: 45000.0 https://www.goplaceit.com/cl/propiedad/arriendo/departamento/la-serena/11733289-departamento-en-arriendo-en-la-serena?&prev=perfil-region
⏳ Progreso: Inspeccionando propiedad 5 de 30...
Descartado precio sospechoso: 45000.0 https://www.goplaceit.com/cl/propiedad/arriendo/departamento/la-serena/12026621-acogedor-departamento-en-arriendo-en-la-serena?&prev=perfil-region
Descartado precio sospechoso: 45000.0 https://www.goplaceit.com/cl/propiedad/arriendo/departamento/la-serena/12551251-departamento-en-arriendo-en-la-serena?&prev=perfil-region
⏳ Progreso: Inspeccionando propiedad 10 de 30...
⏳ Progreso: Inspeccionando propiedad 15 de 30...
⏳ Progreso: Inspeccionando propiedad 20 de 30...
⏳ Progreso: Inspeccionando propiedad 2